# 575 Notebook

In [1]:
import gzip
import json
import pandas as pd
from langchain_core.documents import Document
import re
import string
from nltk.corpus import stopwords
from rank_bm25 import BM25Okapi
import pickle
import os
#import nltk
#nltk.download("stopwords")


In [2]:
# Helper: load first N records from a .jsonl.gz file
def load_jsonl_gz(path, n=200):
    records = []
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return records


# Load small subsets (200 rows)
meta_path = "../data/raw/meta_Appliances.jsonl.gz"
review_path = "../data/raw/Appliances.jsonl.gz"

meta_records = load_jsonl_gz(meta_path, n=200)
review_records = load_jsonl_gz(review_path, n=200)

print(f"Loaded {len(meta_records)} metadata records")
print(f"Loaded {len(review_records)} review records")

# Convert to DataFrames for easier EDA
meta_df = pd.DataFrame(meta_records)
review_df = pd.DataFrame(review_records)

meta_df.head()


Loaded 200 metadata records
Loaded 200 review records


,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,Industrial & Scientific,"ROVSUN Ice Maker Machine Countertop, Make 44lb...",3.7,61,[【Quick Ice Making】This countertop ice machine...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Our Point of View on the Euhomy Ic...,ROVSUN,"[Appliances, Refrigerators, Freezers & Ice Mak...","{'Brand': 'ROVSUN', 'Model Name': 'ICM-2005', ...",B08Z743RRD,None
1,Tools & Home Improvement,"HANSGO Egg Holder for Refrigerator, Deviled Eg...",4.2,75,"[Plastic, Practical Kitchen Storage - Our egg ...",[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': '10 Eggs Egg Holder for Refrigerato...,HANSGO,"[Appliances, Parts & Accessories, Refrigerator...","{'Manufacturer': 'HANSGO', 'Part Number': 'HAN...",B097BQDGHJ,None
2,Tools & Home Improvement,"Clothes Dryer Drum Slide, General Electric, Ho...",3.5,18,[],"[Brand new dryer drum slide, replaces General ...",NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],GE,"[Appliances, Parts & Accessories]","{'Manufacturer': 'RPI', 'Part Number': 'WE1M33...",B00IN9AGAE,None
3,Tools & Home Improvement,154567702 Dishwasher Lower Wash Arm Assembly f...,4.5,26,[MODEL NUMBER:154567702 Dishwasher Lower Wash ...,[MODEL NUMBER:154567702 Dishwasher Lower Wash ...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],folosem,"[Appliances, Parts & Accessories, Dryer Parts ...","{'Manufacturer': 'folosem', 'Part Number': '15...",B0C7K98JZS,None
4,Tools & Home Improvement,Whirlpool W10918546 Igniter,3.8,12,[This is a Genuine OEM Replacement Part.],[Whirlpool Igniter],25.07,[{'thumb': 'https://m.media-amazon.com/images/...,[],Whirlpool,"[Appliances, Parts & Accessories]","{'Manufacturer': 'Whirlpool', 'Part Number': '...",B07QZHQTVJ,None


#### Dataset overview

In [3]:
print("Metadata fields:", meta_df.columns.tolist())
print("Review fields:", review_df.columns.tolist())

print("\nMetadata shape:", meta_df.shape)
print("Review shape:", review_df.shape)


Metadata fields: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together']
Review fields: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']

Metadata shape: (200, 14)
Review shape: (200, 10)


#### Insepct sample records

In [4]:
meta_df.sample(3)

review_df.sample(3)


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
71,5.0,Worked!,Got the dryer working again!,[],B07LG8QYFW,B07LG8QYFW,AHATA6X6MYTC3VNBFJ3WIYVK257A,1628043179921,0,False
167,1.0,"Didn't Work, Unfortunately",Put the pads under the feet of the washer and ...,[],B07QFPKG9V,B07QFPKG9V,AF3I5VHFNDTD3CUFMEPULGWMAMKA,1570235609012,0,True
119,5.0,Good Product,Good Product,[],B00EZACO8M,B00EZACO8M,AFDMZ4TRX3HXQQUGWAHJQTIF65BQ,1617568883512,0,True


#### Langchain

In [6]:
from langchain_core.documents import Document

# Fields we decided to use for retrieval
META_TEXT_FIELDS = ["title", "description", "features"]
REVIEW_TEXT_FIELDS = ["title", "text"]

documents = []

# Process metadata entries
for _, row in meta_df.iterrows():
    text_parts = []

    for field in META_TEXT_FIELDS:
        value = row.get(field)
        if isinstance(value, str) and len(value.strip()) > 0:
            text_parts.append(value.strip())

    # Skip if no meaningful text
    if len(text_parts) == 0:
        continue

    full_text = "\n\n".join(text_parts)

    documents.append(
        Document(
            page_content=full_text,
            metadata={
                "source": "metadata",
                "asin": row.get("parent_asin"),
                "main_category": row.get("main_category"),
                "categories": row.get("categories"),
            }
        )
    )

# Process review entries
for _, row in review_df.iterrows():
    text_parts = []

    for field in REVIEW_TEXT_FIELDS:
        value = row.get(field)
        if isinstance(value, str) and len(value.strip()) > 0:
            text_parts.append(value.strip())

    if len(text_parts) == 0:
        continue

    full_text = "\n\n".join(text_parts)

    documents.append(
        Document(
            page_content=full_text,
            metadata={
                "source": "review",
                "asin": row.get("asin"),
                "rating": row.get("rating"),
                "verified_purchase": row.get("verified_purchase"),
            }
        )
    )

len(documents)


399

#### Selection and justification of fields for retrieval

For retrieval, I focused on fields that contain meaningful natural‑language text.  
From the metadata, **`title`**, **`description`**, and sometimes **`features`** provide the most useful product information.  
From the reviews, the key fields are **`text`** (full review body) and **`title`** (short summary).  
Other fields like price, images, or ratings don’t add much value for text‑based retrieval, so I excluded them.


#### Description of text preprocessing decisions

 I applied light preprocessing to keep the text clean but still semantically rich:  
- lowercase everything  
- remove extra whitespace and any HTML  
- keep punctuation  
- drop empty or extremely short entries  

No stemming or lemmatization, as modern embedding models handle that well on their own.


#### Simple Tokenizer

In [13]:
import re
import string
from nltk.corpus import stopwords

stopwords_set = set(stopwords.words("english"))

def tokenize(text, remove_stopwords=True):
    # lowercase
    text = text.lower()

    # remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # whitespace tokenization
    tokens = text.split()

    if remove_stopwords:
        tokens = [t for t in tokens if t not in stopwords_set]

    return tokens


#### bm25

In [20]:
corpus = [doc.page_content for doc in documents]
tokenized_corpus = [tokenize(text) for text in corpus]

len(tokenized_corpus)

from rank_bm25 import BM25Okapi

bm25 = BM25Okapi(tokenized_corpus)

import pickle
import os

save_dir = "../retrieval_artifacts"
tokenized_path = os.path.join(save_dir, "bm25_tokenized_corpus.pkl")
index_path = os.path.join(save_dir, "bm25_index.pkl")


with open(tokenized_path, "wb") as f:
    pickle.dump(tokenized_corpus, f)

with open(index_path, "wb") as f:
    pickle.dump(bm25, f)

print("Saved BM25 index and tokenized corpus to:", save_dir)


tokenized_path = os.path.join(save_dir, "bm25_tokenized_corpus.pkl")
index_path = os.path.join(save_dir, "bm25_index.pkl")

with open(tokenized_path, "rb") as f:
    tokenized_corpus = pickle.load(f)

with open(index_path, "rb") as f:
    bm25 = pickle.load(f)

print("Loaded BM25 index and corpus from:", save_dir)


Saved BM25 index and tokenized corpus to: ../retrieval_artifacts
Loaded BM25 index and corpus from: ../retrieval_artifacts


#### Query the BM25 Retriever

In [18]:
def bm25_search(query, k=5):
    query_tokens = tokenize(query)
    scores = bm25.get_scores(query_tokens)
    top_k_idx = scores.argsort()[-k:][::-1]

    results = []
    for idx in top_k_idx:
        results.append({
            "score": scores[idx],
            "text": corpus[idx],
            "metadata": documents[idx].metadata
        })
    return results

results = bm25_search("quiet dishwasher with stainless steel interior", k=5)
results


[{'score': np.float64(11.088018495315755),
  'text': 'KitchenAid Superba Series KUDS30SXSS Fully Integrated Dishwasher, Stainless Steel',
  'metadata': {'source': 'metadata',
   'asin': 'B0052EK0MW',
   'main_category': 'Appliances',
   'categories': ['Appliances', 'Dishwashers', 'Built-In Dishwashers']}},
 {'score': np.float64(9.757897726560227),
  'text': 'MLGB Stainless Steel Brushed Pattern Dishwasher Magnet Cover Panel Decal Home Appliance Art, Stainless Steel Fridge Door Cover Decals Magnetic, Black, Mobile Magnetic 23" x 26"',
  'metadata': {'source': 'metadata',
   'asin': 'B09XXFX5SK',
   'main_category': 'Amazon Home',
   'categories': ['Appliances',
    'Parts & Accessories',
    'Dishwasher Parts & Accessories',
    'Hoses']}},
 {'score': np.float64(7.350299200104795),
  'text': 'midea HS-209BESS Beer/Beverage Refrigerator and Dispenser, 5.7 Cubic Feet, Stainless Steel',
  'metadata': {'source': 'metadata',
   'asin': 'B00PLD0YUW',
   'main_category': 'Appliances',
   'cate

#### Implement a keyword‑based retrieval system using BM25

In [19]:
results = bm25_search("quiet dishwasher stainless steel", k=5)
for r in results:
    print(r["score"], r["metadata"], r["text"][:200], "\n")


11.088018495315755 {'source': 'metadata', 'asin': 'B0052EK0MW', 'main_category': 'Appliances', 'categories': ['Appliances', 'Dishwashers', 'Built-In Dishwashers']} KitchenAid Superba Series KUDS30SXSS Fully Integrated Dishwasher, Stainless Steel 

9.757897726560227 {'source': 'metadata', 'asin': 'B09XXFX5SK', 'main_category': 'Amazon Home', 'categories': ['Appliances', 'Parts & Accessories', 'Dishwasher Parts & Accessories', 'Hoses']} MLGB Stainless Steel Brushed Pattern Dishwasher Magnet Cover Panel Decal Home Appliance Art, Stainless Steel Fridge Door Cover Decals Magnetic, Black, Mobile Magnetic 23" x 26" 

7.350299200104795 {'source': 'metadata', 'asin': 'B00PLD0YUW', 'main_category': 'Appliances', 'categories': ['Appliances', 'Refrigerators, Freezers & Ice Makers', 'Kegerators']} midea HS-209BESS Beer/Beverage Refrigerator and Dispenser, 5.7 Cubic Feet, Stainless Steel 

7.2415600656894314 {'source': 'review', 'asin': 'B076VPHFH6', 'rating': 5.0, 'verified_purchase': True} stainle

#### Build embeddings with sentence-transformers

In [21]:
from sentence_transformers import SentenceTransformer

# all-MiniLM-L6-v2
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

corpus = [doc.page_content for doc in documents]

# compute embeddings
embeddings = model.encode(corpus, show_progress_bar=True, convert_to_numpy=True)
embeddings.shape


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/13 [00:00<?, ?it/s]

(399, 384)

#### Build FAISS index

In [22]:
import faiss

dim = embeddings.shape[1]

# simple L2 index
index = faiss.IndexFlatL2(dim)  

# add all vectors
index.add(embeddings)           
print("Number of vectors in index:", index.ntotal)

save_dir = "../semantic_index/"
os.makedirs(save_dir, exist_ok=True)

faiss_index_path = os.path.join(save_dir, "faiss_index.bin")
metadata_path = os.path.join(save_dir, "documents.pkl")

# save FAISS index
faiss.write_index(index, faiss_index_path)

# save documents (for mapping back from index to content/metadata)
with open(metadata_path, "wb") as f:
    pickle.dump(documents, f)

print("Saved FAISS index and documents to:", save_dir)


Number of vectors in index: 399
Saved FAISS index and documents to: ../semantic_index/


#### Load FAISS index and documents

In [23]:
save_dir = "../semantic_index/"
faiss_index_path = os.path.join(save_dir, "faiss_index.bin")
metadata_path = os.path.join(save_dir, "documents.pkl")

index = faiss.read_index(faiss_index_path)

with open(metadata_path, "rb") as f:
    documents = pickle.load(f)

print("Loaded FAISS index and documents from:", save_dir)
print("Index size:", index.ntotal)


Loaded FAISS index and documents from: ../semantic_index/
Index size: 399


#### Semantic Search Function

In [25]:
def semantic_search(query, k=5):
    # embed query
    query_emb = model.encode([query], convert_to_numpy=True)
    
    # search in FAISS
    distances, indices = index.search(query_emb, k)
    
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        doc = documents[idx]
        results.append({
            "score": float(dist),
            "text": doc.page_content,
            "metadata": doc.metadata
        })
    return results

# example
results = semantic_search("quiet stainless steel dishwasher", k=5)
for r in results:
    print("Score:", r["score"])
    print("Metadata:", r["metadata"])
    print("Text snippet:", r["text"][:200], "\n")


Score: 0.7842284440994263
Metadata: {'source': 'metadata', 'asin': 'B0052EK0MW', 'main_category': 'Appliances', 'categories': ['Appliances', 'Dishwashers', 'Built-In Dishwashers']}
Text snippet: KitchenAid Superba Series KUDS30SXSS Fully Integrated Dishwasher, Stainless Steel 

Score: 0.9125173687934875
Metadata: {'source': 'metadata', 'asin': 'B00T5K1924', 'main_category': nan, 'categories': ['Appliances', 'Dishwashers', 'Built-In Dishwashers']}
Text snippet: Frigidaire Professional FPID2495QF Fully Integrated Dishwasher 

Score: 0.9262876510620117
Metadata: {'source': 'review', 'asin': 'B0047PF5MM', 'rating': 5.0, 'verified_purchase': True}
Text snippet: Finally, the branded product to clean my Miele!

I have been trying to find something that cleans my 6 year old dish washer and in the past it was run the machine empty.  Although we have never notice 

Score: 0.939934492111206
Metadata: {'source': 'review', 'asin': 'B09LM3ZW5N', 'rating': 5.0, 'verified_purchase': False}
Text snippe